In [ ]:
# This script builds a text‑based classifier from the manually verified points
# (columns: name, keyword, original_layer, verified_type). It then applies
# the classifier to all points in the three layers (both, depot_only, filling_only)
# and adds columns 'predicted_type' and 'confidence' (max probability) to the GeoPackage.
# If a verified_type column exists, it also reports accuracy and confusion matrix.
# The model and vectorizer are saved to disk so they can be reused on new data.
#model achieve accuracy of above 93% if confidence is above 0.7

#TO BE CHECK AND IMPROVED, NOT POSSIBLE TO USE NOW

import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

DATA_DIR = "dataset_big"
gpkg = f"{DATA_DIR}/lpg_infrastructure_3857.gpkg"
model_file = f"{DATA_DIR}/lpg_classifier_model.joblib"
vectorizer_file = f"{DATA_DIR}/lpg_vectorizer.joblib"

layers = ["both", "depot_only", "filling_only"]

# Load and combine layers
dfs = []
for layer in layers:
    gdf = gpd.read_file(gpkg, layer=layer)
    gdf["original_layer"] = layer
    dfs.append(gdf)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.reset_index(drop=True)

# Prepare feature text from name, keyword, and original_layer
combined["text"] = (
    combined["name"].fillna("") + " " +
    combined["keyword"].fillna("") + " " +
    combined["original_layer"].fillna("")
)

# Check if we have verified labels
has_labels = "verified_type" in combined.columns and combined["verified_type"].notna().any()


if has_labels:
    # Use points with non‑null verified_type as training data
    train_data = combined[combined["verified_type"].notna()].copy()
    X_train = train_data["text"]
    y_train = train_data["verified_type"]
    print(f"Training on {len(X_train)} labeled points")
    print("Class distribution:")
    print(y_train.value_counts())

    # Vectorize
    vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
    X_vec = vectorizer.fit_transform(X_train)

    # Cross‑validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    # Exclude classes with < 2 samples from CV
    min_class_size = 2
    valid_classes = y_train.value_counts()[y_train.value_counts() >= min_class_size].index
    mask = y_train.isin(valid_classes).values
    X_cv = X_vec[mask]
    y_cv = y_train[mask]
    if X_cv.shape[0] > 0:
        clf_cv = LogisticRegression(max_iter=1000)
        scores = cross_val_score(clf_cv, X_cv, y_cv, cv=cv, scoring="accuracy")
        print(f"Cross‑validation accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

    # Train final model on all labeled data
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_vec, y_train)

    # Evaluate on training set (for information)
    y_pred = clf.predict(X_vec)
    print("\nClassification report (training set):")
    print(classification_report(y_train, y_pred, zero_division=0))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_train, y_pred))

    # Save model and vectorizer
    joblib.dump(clf, model_file)
    joblib.dump(vectorizer, vectorizer_file)
    print(f"\nModel saved to {model_file} and {vectorizer_file}")

else:
    # No labels – try to load previously saved model
    if os.path.exists(model_file) and os.path.exists(vectorizer_file):
        print("No verified_type found. Loading previously saved model...")
        clf = joblib.load(model_file)
        vectorizer = joblib.load(vectorizer_file)
    else:
        raise FileNotFoundError("No verified_type column and no saved model found. Cannot proceed.")

# Predict on all points
X_all = vectorizer.transform(combined["text"])
proba = clf.predict_proba(X_all)
predicted = clf.predict(X_all)
confidence = np.max(proba, axis=1)

# Add columns to the GeoDataFrame (keep geometry)
combined["predicted_type"] = predicted
combined["confidence"] = confidence

# Optional: flag low confidence (e.g., < 0.6)
combined["low_confidence"] = confidence < 0.6

# Save each layer back to the GeoPackage
for layer in layers:
    layer_gdf = combined[combined["original_layer"] == layer].copy()
    # Drop temporary columns before saving (keep geometry)
    cols_to_keep = [col for col in layer_gdf.columns if col not in ["text", "original_layer"]]
    layer_gdf = layer_gdf[cols_to_keep]
    # Reproject to 3857 (already in that CRS, but ensure)
    if layer_gdf.crs is None:
        layer_gdf = layer_gdf.set_crs("EPSG:3857")
    # Write back (overwrite existing layer)
    layer_gdf.to_file(gpkg, layer=layer, driver="GPKG")
    print(f"Saved layer '{layer}' with predicted_type and confidence.")

print("\nDone. All layers updated with predicted_type and confidence.")

Training on 670 labeled points
Class distribution:
verified_type
filling plant    376
invalid          190
retailer          90
primary depot     14
Name: count, dtype: int64
Cross‑validation accuracy: 0.610 ± 0.039

Classification report (training set):
               precision    recall  f1-score   support

filling plant       0.69      0.97      0.80       376
      invalid       0.81      0.56      0.66       190
primary depot       1.00      0.14      0.25        14
     retailer       1.00      0.08      0.14        90

     accuracy                           0.72       670
    macro avg       0.87      0.44      0.47       670
 weighted avg       0.77      0.72      0.66       670


Confusion matrix:
[[364  12   0   0]
 [ 83 107   0   0]
 [  6   6   2   0]
 [ 76   7   0   7]]

Model saved to lpg_classifier_model.joblib and lpg_vectorizer.joblib
Saved layer 'both' with predicted_type and confidence.
Saved layer 'depot_only' with predicted_type and confidence.
Saved layer 'filling

In [ ]:
# This cell evaluates the trained model on the labeled data,
# showing how many points reach different confidence thresholds
# and what accuracy they achieve. Useful for deciding a cutoff
# for automatic classification.

import geopandas as gpd
import pandas as pd
import numpy as np
import joblib

DATA_DIR = "dataset_big"
gpkg = f"{DATA_DIR}/lpg_infrastructure_3857.gpkg"
model_file = f"{DATA_DIR}/lpg_classifier_model.joblib"
vectorizer_file = f"{DATA_DIR}/lpg_vectorizer.joblib"

layers = ["both", "depot_only", "filling_only"]

dfs = []
for layer in layers:
    gdf = gpd.read_file(gpkg, layer=layer)
    gdf["original_layer"] = layer
    dfs.append(gdf)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.reset_index(drop=True)

# Load model and vectorizer
clf = joblib.load(model_file)
vectorizer = joblib.load(vectorizer_file)

# Prepare text (same as during training)
combined["text"] = (
    combined["name"].fillna("") + " " +
    combined["keyword"].fillna("") + " " +
    combined["original_layer"].fillna("")
)

X_all = vectorizer.transform(combined["text"])
proba = clf.predict_proba(X_all)
predicted = clf.predict(X_all)
confidence = np.max(proba, axis=1)

# Add columns to combined
combined["predicted_type"] = predicted
combined["confidence"] = confidence

# If we have verified_type, evaluate
if "verified_type" in combined.columns and combined["verified_type"].notna().any():
    y_true = combined["verified_type"]
    y_pred = combined["predicted_type"]

    print("Evaluation of model on all labeled points (optimistic, training set).")
    print(f"Total labeled points: {len(y_true)}")

    thresholds = [0.6, 0.7, 0.8, 0.9]
    print("\nConfidence threshold | Points above | Accuracy among them")
    print("-" * 60)
    for thresh in thresholds:
        mask = combined["confidence"] >= thresh
        n_points = mask.sum()
        if n_points == 0:
            print(f"{thresh:<20} | {n_points:<12} | N/A")
            continue
        acc = (y_true[mask] == y_pred[mask]).mean()
        print(f"{thresh:<20} | {n_points:<12} | {acc:.3f}")

    # Show breakdown per class for threshold 0.8
    thresh = 0.8
    mask = combined["confidence"] >= thresh
    if mask.any():
        print(f"\nFor confidence >= {thresh}:")
        print("True class distribution among these points:")
        print(y_true[mask].value_counts())
        print("\nConfusion matrix (rows true, cols predicted):")
        cm = pd.crosstab(y_true[mask], y_pred[mask])
        print(cm)
    else:
        print(f"\nNo points with confidence >= {thresh}.")

else:
    print("No verified_type found. Cannot evaluate accuracy.")

Evaluation of model on all labeled points (optimistic, training set).
Total labeled points: 670

Confidence threshold | Points above | Accuracy among them
------------------------------------------------------------
0.6                  | 352          | 0.918
0.7                  | 159          | 0.943
0.8                  | 22           | 0.955
0.9                  | 0            | N/A

For confidence >= 0.8:
True class distribution among these points:
verified_type
filling plant    21
invalid           1
Name: count, dtype: int64

Confusion matrix (rows true, cols predicted):
predicted_type  filling plant
verified_type                
filling plant              21
invalid                     1
